# Decision Tree: FARS Person-Level Data

## Load the data

Reads the person-level table saved by `load_fars_year.ipynb` from `processed/fars_<YEAR>_person.<LOAD_FORMAT>`. Run that notebook first for the year you want.

In [23]:
from pathlib import Path
import pandas as pd

YEAR = 2016
LOAD_FORMAT = "csv"   # "parquet" or "csv"
PROCESSED_DIR = Path("processed")

in_path = PROCESSED_DIR / f"fars_{YEAR}_person.{LOAD_FORMAT}"

if LOAD_FORMAT == "parquet":
    df = pd.read_parquet(in_path)
elif LOAD_FORMAT == "csv":
    df = pd.read_csv(in_path, low_memory=False)
else:
    raise ValueError('LOAD_FORMAT must be "parquet" or "csv"')

print(f"Loaded {in_path}: {df.shape[0]:,} people x {df.shape[1]} columns")
#df.head()

Loaded processed/fars_2016_person.csv: 86,474 people x 410 columns


In [24]:
ID_COLS = ["ST_CASE", "VEH_NO", "PER_NO", "YEAR", "STATE", "INJ_SEV"]  # key, split key, label source

BASE_COLS = [
    "AGE", "SEX", "PER_TYP", "SEAT_POS", "REST_USE", "BODY_TYP", "MOD_YEAR",
    "NUMOCCS", "TOW_VEH", "L_STATUS", "VSPD_LIM", "VALIGN", "VPROFILE",
    "VSURCOND", "VTRAFWAY", "VNUM_LAN", "VTRAFCON", "VTCONT_F", "HOUR",
    "DAY_WEEK", "MONTH", "RUR_URB", "FUNC_SYS", "LGT_COND", "WEATHER",
    "RELJCT2",
]
IMPACT_COLS = [
    "IMPACT1", "P_CRASH1", "P_CRASH2", "PCRASH4", "PCRASH5", "SPEEDREL",
    "DR_DRINK", "HARM_EV", "MAN_COLL", "REL_ROAD", "VE_FORMS", "PEDS",
    "PERNOTMVIT", "PERSONS", "PERMVIT", "DRUNK_DR",
]
POST_COLS = [
    "AIR_BAG", "EJECTION", "EXTRICAT", "ROLLOVER", "DEFORMED", "FIRE_EXP",
]

base_df             = df[ID_COLS + BASE_COLS]
base_impact_df      = df[ID_COLS + BASE_COLS + IMPACT_COLS]
base_impact_post_df = df[ID_COLS + BASE_COLS + IMPACT_COLS + POST_COLS]

In [25]:
print(f"BASE:               {len(BASE_COLS)} features")
print(f"BASE+IMPACT:        {len(BASE_COLS + IMPACT_COLS)} features")
print(f"BASE+IMPACT+POST:   {len(BASE_COLS + IMPACT_COLS + POST_COLS)} features")

BASE:               26 features
BASE+IMPACT:        42 features
BASE+IMPACT+POST:   48 features


In [26]:
FULL_SET = df[ID_COLS + BASE_COLS + IMPACT_COLS + POST_COLS].copy()

## PREPROCESSING STEP 1, Include binary output FATAL

In [27]:
# Keep only known injury levels 0-4 (drop 5 = severity unknown, 6 = died before crash, 9 = unknown)
FULL_SET = FULL_SET[FULL_SET["INJ_SEV"].between(0, 4)]

# Label: 1 = fatal (INJ_SEV 4), 0 = not fatal
FULL_SET["FATAL"] = (FULL_SET["INJ_SEV"] == 4).astype(int)

print(f"{len(FULL_SET):,} people remain")
FULL_SET["FATAL"].value_counts().sort_index()

84,823 people remain


FATAL
0    47017
1    37806
Name: count, dtype: int64

## PREPROCESS STEP 2, keep only vehicle occupants in dataset

In [28]:
# Keep only vehicle occupants: PER_TYP 1 = driver, 2 = passenger
FULL_SET = FULL_SET[FULL_SET["PER_TYP"].isin([1, 2])]

print(f"{len(FULL_SET):,} people remain")

76,695 people remain


## PREPROCESS STEP 3, one-hot encoding

In [29]:
# True numbers; everything else is a category code
NUMERIC_COLS = ["AGE", "MOD_YEAR", "NUMOCCS", "VSPD_LIM", "VE_FORMS",
                "PEDS", "PERNOTMVIT", "PERSONS", "PERMVIT"]
CATEGORICAL_COLS = [c for c in BASE_COLS + IMPACT_COLS + POST_COLS if c not in NUMERIC_COLS]

# One 0/1 column per code (e.g. BODY_TYP_4); unknown codes like 99 get their own column
FULL_SET[CATEGORICAL_COLS] = FULL_SET[CATEGORICAL_COLS].astype(int)
FULL_SET = pd.get_dummies(FULL_SET, columns=CATEGORICAL_COLS, dtype=int)

print(f"{FULL_SET.shape[1]} columns after one-hot encoding")

551 columns after one-hot encoding


## PREPROCESS STEP 4, remove rows with unknown values in the numeric columns

Crash data specifically, why drop? 
* NHTSA: NHTSA itself uses multiple imputation in FARS, to fill in missing blood alcohol concentration (BAC) values. Their technical reports document the method, which is a strong precedent that FARS unknowns matter.
* Jahan, Bhowmik, Hoover & Eluru (Transportation Research Record): in a crash-severity model on national CRSS data, multiple imputation fit better than dropping variables. Their simulations also found that dropping rows can be preferable when the sample is large and the missing share is small. That’s exactly your situation: 73,000 rows with 4.8% missing.
* Adediji & Noland (2020, Findings): imputing missing traffic volumes in a crash-count model noticeably shifted the coefficients. So imputation isn’t automatically safe either.


In [30]:
# FARS "unknown" codes in the numeric columns
UNKNOWN_CODES = {
    "AGE":      [998, 999],
    "MOD_YEAR": [9998, 9999],
    "NUMOCCS":  [97, 98, 99],
    "VSPD_LIM": [0, 98, 99],   # 0 = no statutory limit, treated as unknown
}

# Drop any row that has an unknown code in one of these columns
for col, codes in UNKNOWN_CODES.items():
    FULL_SET = FULL_SET[~FULL_SET[col].isin(codes)]

print(f"{len(FULL_SET):,} people remain")

72,525 people remain


In [31]:
# Save the preprocessed dataset (unscaled: trees do not need scaling)
out_path = Path("processed/decision_tree") / f"full_set_preprocessed_unscaled_{YEAR}.csv"
FULL_SET.to_csv(out_path, index=False)

print(f"Saved {out_path}: {FULL_SET.shape[0]:,} rows x {FULL_SET.shape[1]} columns")

Saved processed/decision_tree/full_set_preprocessed_unscaled_2016.csv: 72,525 rows x 551 columns


In [32]:
import re

# Model-ready feature lists for each tier, matched to the encoded column names
def encoded_cols(cols):
    out = []
    for col in cols:
        if col in NUMERIC_COLS:
            out.append(col)
        else:
            out += [c for c in FULL_SET.columns if re.fullmatch(rf"{col}_\d+", c)]
    return out

BASE_FEATURES             = encoded_cols(BASE_COLS)
BASE_IMPACT_FEATURES      = encoded_cols(BASE_COLS + IMPACT_COLS)
BASE_IMPACT_POST_FEATURES = encoded_cols(BASE_COLS + IMPACT_COLS + POST_COLS)

print(f"BASE:               {len(BASE_FEATURES)} columns")
print(f"BASE+IMPACT:        {len(BASE_IMPACT_FEATURES)} columns")
print(f"BASE+IMPACT+POST:   {len(BASE_IMPACT_POST_FEATURES)} columns")

# Usage:  X = FULL_SET[BASE_FEATURES];  y = FULL_SET["FATAL"];  groups = FULL_SET["STATE"]

BASE:               300 columns
BASE+IMPACT:        511 columns
BASE+IMPACT+POST:   544 columns


## DATA READY! Start here.

In [33]:
# ML-ready, processed datasets: IDs + label + that tier's features
base_df             = FULL_SET[ID_COLS + ["FATAL"] + BASE_FEATURES]
base_impact_df      = FULL_SET[ID_COLS + ["FATAL"] + BASE_IMPACT_FEATURES]
base_impact_post_df = FULL_SET[ID_COLS + ["FATAL"] + BASE_IMPACT_POST_FEATURES]